# TraceGuard AI — Vector Search Experiments + Hybrid RAG + Traceability-Aware Impact Analysis

This notebook extends the working **02-basic-rag.ipynb** pipeline with controlled Module 2 vector-search experiments while preserving the downstream TraceGuard impact-analysis flow.

## Objectives

1. Preserve lexical retrieval as a baseline.
2. Preserve the existing brute-force cosine semantic retrieval as the semantic baseline.
3. Evaluate `minsearch.VectorSearch` using the same embeddings and artifact population.
4. Evaluate persistent `sqlitesearch.VectorSearchIndex` retrieval.
5. Optionally compare embedding models.
6. Compare lexical, brute-force vector, Minsearch vector, SQLite vector, and hybrid retrieval against the existing evaluation ground truth.
7. Measure Recall@K, Precision@K, indexing time, and query latency.
8. Keep traceability expansion, candidate ranking, LLM grounding, validation, Release mapping, and Baseline determination unchanged.

> **Architecture rule:** vector similarity is retrieval evidence, not an impact decision. TraceGuard continues to use explicit traceability and grounded LLM assessment downstream.

> **Evaluation rule:** the best retrieval architecture is selected from measured results rather than assumed in advance.


## 1. Imports and Paths


In [ ]:
from pathlib import Path
from collections import defaultdict, deque
import ast
import json
import math
import re
import time
import os

import numpy as np
import pandas as pd
import minsearch
from minsearch import Index
from sentence_transformers import SentenceTransformer

# Module 2 vector-search dependencies.
# Recommended environment commands:
#   pip install -U "minsearch>=0.2.0" "sqlitesearch>=0.3.0" sentence-transformers
#
# The notebook keeps optional experiments isolated so the original TraceGuard
# pipeline can still run if sqlitesearch is not installed.
try:
    from minsearch import VectorSearch
    MINSEARCH_VECTOR_AVAILABLE = True
except ImportError:
    VectorSearch = None
    MINSEARCH_VECTOR_AVAILABLE = False

try:
    from sqlitesearch import VectorSearchIndex
    SQLITESEARCH_AVAILABLE = True
except ImportError:
    VectorSearchIndex = None
    SQLITESEARCH_AVAILABLE = False

data_path = Path("../data")
vector_db_path = data_path / "traceguard_vectors.db"

print("Data path:", data_path.resolve())
print("Minsearch VectorSearch available:", MINSEARCH_VECTOR_AVAILABLE)
print("sqlitesearch available:", SQLITESEARCH_AVAILABLE)


## 2. Load Datasets


In [ ]:
artifacts_df = pd.read_csv(data_path / "artifacts.csv")
baselines_df = pd.read_csv(data_path / "baselines.csv")
evaluation_df = pd.read_csv(data_path / "evaluation_new_crs.csv")
evaluation_ground_truth_df = pd.read_csv(data_path / "evaluation_ground_truth.csv")

print("Artifacts:", len(artifacts_df))
print("Baselines:", len(baselines_df))
print("Evaluation CRs:", len(evaluation_df))
print("Evaluation ground-truth rows:", len(evaluation_ground_truth_df))
print("Artifact columns:", artifacts_df.columns.tolist())
print("Baseline columns:", baselines_df.columns.tolist())

artifacts_df.head()


## 3. Prepare Searchable Content


In [ ]:
for col in ["Summary", "Text", "Spawns", "Covers", "State", "Project", "Document_ID"]:
    if col not in artifacts_df.columns:
        artifacts_df[col] = ""
    artifacts_df[col] = artifacts_df[col].fillna("").astype(str)

artifacts_df["ID"] = artifacts_df["ID"].astype(str)
artifacts_df["Type"] = artifacts_df["Type"].fillna("").astype(str)

artifacts_df["Search_Text"] = (
    artifacts_df["ID"].str.strip() + " | " +
    artifacts_df["Type"].str.strip() + " | " +
    artifacts_df["Summary"].str.strip() + " | " +
    artifacts_df["Text"].str.strip()
).str.strip(" |")

print("Total artifacts:", len(artifacts_df))
print("With searchable content:", (artifacts_df["Search_Text"] != "").sum())
print("Artifact types:")
display(artifacts_df["Type"].value_counts().to_frame("count"))


## 4. Configure Independent Retrieval by Artifact Type

Retrieval breadth is now configurable **per artifact type**. These are baseline candidate-pool settings, not final tuned values. Historical evaluation should determine the appropriate size for each lifecycle type.

Semantic scoring and Top-K retention are separate: every artifact receives a semantic score for evaluation, while only the configured retained pool proceeds into candidate generation.


In [ ]:
important_types = [
    "Change Request",
    "Problem Report",
    "ALM Input",
    "ALM Requirement",
    "ALM Specification",
    "ALM Test Suite",
    "ALM Test Case",
    "Task",
    "Release",
]

available_types = set(artifacts_df["Type"].unique())
retrieval_types = [t for t in important_types if t in available_types]

# Baseline values only. Tune these from historical Recall@K / precision evaluation.
PER_TYPE_CANDIDATES = {
    "Change Request": 50,
    "Problem Report": 50,
    "ALM Input": 50,
    "ALM Requirement": 50,
    "ALM Specification": 50,
    "ALM Test Suite": 50,
    "ALM Test Case": 50,
    "Task": 50,
    "Release": 50,
}
DEFAULT_PER_TYPE_CANDIDATES = 50

# Completely separate downstream LLM budget.
LLM_CONTEXT_LIMIT = 80

# Reserve context space for similarity-discovered artifacts with no ALM link.
UNLINKED_CONTEXT_FRACTION = 0.25

print("Configured retrieval types / retained candidate limits:")
for t in retrieval_types:
    print(
        f" - {t}: {int((artifacts_df['Type'] == t).sum())} total, "
        f"retain up to {PER_TYPE_CANDIDATES.get(t, DEFAULT_PER_TYPE_CANDIDATES)}"
    )


## 5. Incoming Change


In [ ]:
EVALUATION_CR_ID = None

sample_query = """
Improve battery state of charge monitoring to handle invalid or implausible
signal conditions and improve diagnostic recovery.
""".strip()

if EVALUATION_CR_ID:
    eval_match = evaluation_df[
        evaluation_df["Evaluation_CR_ID"].astype(str) == str(EVALUATION_CR_ID)
    ]

    if eval_match.empty:
        raise ValueError(f"Unknown Evaluation_CR_ID: {EVALUATION_CR_ID}")

    query = str(eval_match.iloc[0]["Summary"]).strip()
    print("Evaluation CR:", EVALUATION_CR_ID)
else:
    query = sample_query

print("Incoming CR / change description:")
print(query)


## 6. Build Per-Type Lexical Indexes


In [ ]:
document_columns = [
    "ID", "Document_ID", "Type", "Summary", "Text",
    "State", "Project", "Spawns", "Covers", "Search_Text"
]

lexical_indexes = {}
documents_by_type = {}

for artifact_type in retrieval_types:
    type_df = artifacts_df[artifacts_df["Type"] == artifact_type].copy()
    docs = type_df[document_columns].fillna("").to_dict(orient="records")
    if not docs:
        continue

    idx = Index(
        text_fields=["Search_Text"],
        keyword_fields=["ID", "Document_ID", "State", "Project", "Type"]
    )
    idx.fit(docs)

    lexical_indexes[artifact_type] = idx
    documents_by_type[artifact_type] = docs

print("Per-type lexical indexes:", len(lexical_indexes))


## 7. Baseline Semantic Retrieval — Brute-Force Cosine Search

This is the semantic baseline inherited from `02-basic-rag.ipynb`.

All artifacts are embedded with `sentence-transformers/all-MiniLM-L6-v2`. Embeddings are normalized, so the dot product used during retrieval is cosine similarity. Every artifact receives a score and a full semantic rank; Top-K retention happens only afterward.

The baseline is intentionally preserved so Module 2 vector-search implementations can be compared against exactly the same embedding representation.


In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

embedding_cache = {}

for artifact_type in retrieval_types:
    type_df = artifacts_df[artifacts_df["Type"] == artifact_type].copy()
    texts = type_df["Search_Text"].tolist()
    if not texts:
        continue

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    embedding_cache[artifact_type] = {
        "ids": type_df["ID"].tolist(),
        "embeddings": np.asarray(embeddings),
    }

print("Semantic embedding sets:", len(embedding_cache))


## 8. Hybrid Per-Type Retrieval: Score All, Then Retain Configurable Top-K

Lexical and semantic evidence remain separate. Semantic evaluation uses the complete scored population; candidate generation retains only the configurable per-type pool.


In [ ]:
def lexical_retrieve_by_type(query_text, artifact_type, limit):
    idx = lexical_indexes.get(artifact_type)
    if idx is None:
        return []

    results = idx.search(
        query=query_text,
        num_results=min(limit, len(documents_by_type[artifact_type]))
    )

    output = []
    for rank, item in enumerate(results, start=1):
        row = dict(item)
        raw_score = row.get("_score", row.get("score", np.nan))
        row["lexical_score"] = float(raw_score) if pd.notna(raw_score) else np.nan
        row["lexical_rank"] = rank
        row["retrieval_source"] = "Lexical retrieval"
        output.append(row)
    return output


def score_all_semantic_by_type(query_text, artifact_type):
    """Return every artifact of this type with cosine similarity and full-population rank."""
    cached = embedding_cache.get(artifact_type)
    if cached is None:
        return pd.DataFrame()

    query_embedding = embedding_model.encode(
        [query_text],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    similarities = cached["embeddings"] @ query_embedding
    order = np.argsort(-similarities)

    rows = []
    for rank, pos in enumerate(order, start=1):
        artifact_id = str(cached["ids"][pos])
        matches = artifacts_df.loc[artifacts_df["ID"] == artifact_id]
        row = matches.iloc[0].to_dict()
        result = dict(row)
        result["semantic_similarity"] = float(similarities[pos])
        result["semantic_rank"] = rank
        rows.append(result)

    return pd.DataFrame(rows)


# Complete semantic scoring is preserved for evaluation BEFORE Top-K.
all_semantic_scores_by_type = {}
semantic_results_by_type = {}
lexical_results_by_type = {}

for artifact_type in retrieval_types:
    all_scored = score_all_semantic_by_type(query, artifact_type)
    all_semantic_scores_by_type[artifact_type] = all_scored

    retain_k = PER_TYPE_CANDIDATES.get(
        artifact_type, DEFAULT_PER_TYPE_CANDIDATES
    )

    semantic_results_by_type[artifact_type] = (
        all_scored.head(retain_k).assign(retrieval_source="Semantic similarity")
        .to_dict(orient="records")
    )

    lexical_results_by_type[artifact_type] = lexical_retrieve_by_type(
        query, artifact_type, retain_k
    )

# Convenient conceptual outputs.
similar_crs = semantic_results_by_type.get("Change Request", [])
similar_prs = semantic_results_by_type.get("Problem Report", [])
similar_requirements = semantic_results_by_type.get("ALM Requirement", [])
similar_specifications = semantic_results_by_type.get("ALM Specification", [])
similar_tests = (
    semantic_results_by_type.get("ALM Test Case", [])
    + semantic_results_by_type.get("ALM Test Suite", [])
)
similar_tasks = semantic_results_by_type.get("Task", [])
similar_releases = semantic_results_by_type.get("Release", [])

for artifact_type in retrieval_types:
    print(
        f"{artifact_type}: "
        f"{len(all_semantic_scores_by_type[artifact_type])} semantic scores available / "
        f"{len(semantic_results_by_type[artifact_type])} semantic retained / "
        f"{len(lexical_results_by_type[artifact_type])} lexical retained"
    )


## 8A. Module 2 Experiment — Minsearch VectorSearch

`minsearch.VectorSearch` provides an in-memory vector-search abstraction over pre-computed embeddings. Because it uses cosine similarity over the same vectors, its retrieval quality should normally be close to the brute-force baseline; the experiment verifies that assumption on TraceGuard data.

Indexes remain **per artifact type**, preserving TraceGuard's lifecycle-balanced retrieval design.


### 8A.1 Installed API Verification Gate

This cell verifies the package versions and callable signatures in the **same Python environment that executes this notebook**. The optional Minsearch and SQLite experiments are not trusted merely because their imports succeed.

Later cells also perform a real one-result build/search smoke test. A backend that fails is excluded from evaluation instead of hard-failing the rest of TraceGuard.


In [ ]:
import inspect
import importlib.metadata as importlib_metadata

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"

def safe_signature(obj):
    try:
        return inspect.signature(obj)
    except (TypeError, ValueError):
        return None

def parameter_names(obj):
    sig = safe_signature(obj)
    return set(sig.parameters) if sig is not None else set()

print("minsearch version:", package_version("minsearch"))
print("sqlitesearch version:", package_version("sqlitesearch"))

print("\nminsearch.VectorSearch available:", MINSEARCH_VECTOR_AVAILABLE)
if MINSEARCH_VECTOR_AVAILABLE:
    print("constructor:", safe_signature(VectorSearch))
    print("fit:", safe_signature(VectorSearch.fit))
    print("search:", safe_signature(VectorSearch.search))

print("\nsqlitesearch.VectorSearchIndex available:", SQLITESEARCH_AVAILABLE)
if SQLITESEARCH_AVAILABLE:
    print("constructor:", safe_signature(VectorSearchIndex))
    print("fit:", safe_signature(VectorSearchIndex.fit))
    print("search:", safe_signature(VectorSearchIndex.search))

MINSEARCH_CTOR_PARAMS = parameter_names(VectorSearch) if MINSEARCH_VECTOR_AVAILABLE else set()
MINSEARCH_SEARCH_PARAMS = parameter_names(VectorSearch.search) if MINSEARCH_VECTOR_AVAILABLE else set()
SQLITE_CTOR_PARAMS = parameter_names(VectorSearchIndex) if SQLITESEARCH_AVAILABLE else set()
SQLITE_SEARCH_PARAMS = parameter_names(VectorSearchIndex.search) if SQLITESEARCH_AVAILABLE else set()

print("\nInstalled API inspection complete.")


In [ ]:
minsearch_vector_indexes = {}
minsearch_vector_build_seconds = np.nan
MINSEARCH_VECTOR_SMOKE_OK = False
MINSEARCH_VECTOR_ERROR = None

if MINSEARCH_VECTOR_AVAILABLE:
    try:
        t0 = time.perf_counter()

        for artifact_type in retrieval_types:
            cached = embedding_cache.get(artifact_type)
            if cached is None:
                continue

            type_df = artifacts_df[artifacts_df["Type"] == artifact_type].copy().reset_index(drop=True)
            payload = type_df[document_columns].fillna("").to_dict(orient="records")

            ctor_kwargs = {}
            if "keyword_fields" in MINSEARCH_CTOR_PARAMS:
                ctor_kwargs["keyword_fields"] = ["Type", "State", "Project"]

            idx = VectorSearch(**ctor_kwargs)
            idx.fit(np.asarray(cached["embeddings"], dtype=np.float32), payload)
            minsearch_vector_indexes[artifact_type] = idx

        minsearch_vector_build_seconds = time.perf_counter() - t0

        smoke_type = next(iter(minsearch_vector_indexes))
        smoke_q = embedding_model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]

        search_kwargs = {}
        if "num_results" in MINSEARCH_SEARCH_PARAMS:
            search_kwargs["num_results"] = 1
        elif "limit" in MINSEARCH_SEARCH_PARAMS:
            search_kwargs["limit"] = 1
        else:
            raise TypeError("Installed VectorSearch.search exposes neither num_results nor limit.")

        smoke = minsearch_vector_indexes[smoke_type].search(
            np.asarray(smoke_q, dtype=np.float32), **search_kwargs
        )
        if len(smoke) < 1:
            raise RuntimeError("Minsearch smoke test returned no result.")

        MINSEARCH_VECTOR_SMOKE_OK = True
        print("Minsearch build/search smoke test: PASS")
        print(f"Build time: {minsearch_vector_build_seconds:.4f} seconds")

    except Exception as exc:
        MINSEARCH_VECTOR_ERROR = repr(exc)
        minsearch_vector_indexes = {}
        print("Minsearch build/search smoke test: FAIL")
        print(MINSEARCH_VECTOR_ERROR)
        print("Minsearch will be excluded from evaluation.")
else:
    print("Minsearch VectorSearch unavailable; experiment skipped.")


In [ ]:
def minsearch_vector_retrieve_by_type(query_text, artifact_type, limit, model=embedding_model):
    if not MINSEARCH_VECTOR_SMOKE_OK:
        return []

    idx = minsearch_vector_indexes.get(artifact_type)
    if idx is None:
        return []

    qv = model.encode([query_text], normalize_embeddings=True, show_progress_bar=False)[0]

    search_kwargs = {}
    if "num_results" in MINSEARCH_SEARCH_PARAMS:
        search_kwargs["num_results"] = min(limit, len(documents_by_type.get(artifact_type, [])))
    elif "limit" in MINSEARCH_SEARCH_PARAMS:
        search_kwargs["limit"] = min(limit, len(documents_by_type.get(artifact_type, [])))
    else:
        return []

    results = idx.search(np.asarray(qv, dtype=np.float32), **search_kwargs)

    output = []
    for rank, item in enumerate(results, start=1):
        row = dict(item)
        raw_score = row.get("score", row.get("_score", np.nan))
        row["vector_similarity"] = float(raw_score) if pd.notna(raw_score) else np.nan
        row["vector_rank"] = rank
        row["retrieval_source"] = "Minsearch vector"
        output.append(row)
    return output

minsearch_vector_results_by_type = {}
minsearch_query_seconds = np.nan

if MINSEARCH_VECTOR_SMOKE_OK:
    t0 = time.perf_counter()
    for artifact_type in retrieval_types:
        retain_k = PER_TYPE_CANDIDATES.get(artifact_type, DEFAULT_PER_TYPE_CANDIDATES)
        minsearch_vector_results_by_type[artifact_type] = minsearch_vector_retrieve_by_type(
            query, artifact_type, retain_k
        )
    minsearch_query_seconds = time.perf_counter() - t0
    print(f"Current-query Minsearch retrieval: {minsearch_query_seconds:.4f} seconds")
else:
    print("Minsearch retrieval skipped: smoke test did not pass.")


## 8B. Module 2 Experiment — Persistent SQLite Vector Search

`sqlitesearch.VectorSearchIndex` persists vector indexes to a local SQLite database. This separates **corpus ingestion/index construction** from **query-time retrieval**.

For TraceGuard, the architectural question is different from the accuracy question:

- Retrieval metrics tell us whether the index preserves useful affected-artifact recall.
- Build/load/query timings tell us whether persistence improves the runtime architecture.
- Persistence should not be described as an accuracy improvement unless the measured evaluation demonstrates one.

The notebook uses one persistent database file and a separate vector index per artifact type.


In [ ]:
REBUILD_SQLITE_VECTOR_INDEX = True

sqlite_vector_indexes = {}
sqlite_vector_build_seconds = np.nan
SQLITE_VECTOR_SMOKE_OK = False
SQLITE_VECTOR_ERROR = None

def safe_index_name(artifact_type):
    return re.sub(r"[^A-Za-z0-9]+", "_", artifact_type).strip("_").lower()

def make_sqlite_index(artifact_type):
    kwargs = {}
    if "keyword_fields" in SQLITE_CTOR_PARAMS:
        kwargs["keyword_fields"] = ["Type", "State", "Project"]
    if "id_field" in SQLITE_CTOR_PARAMS:
        kwargs["id_field"] = "ID"

    index_name = f"traceguard_{safe_index_name(artifact_type)}"

    # Shared DB only when this installed release exposes index_name.
    db_file = vector_db_path if "index_name" in SQLITE_CTOR_PARAMS else data_path / f"{index_name}.db"

    if "index_name" in SQLITE_CTOR_PARAMS:
        kwargs["index_name"] = index_name
    if "db_path" in SQLITE_CTOR_PARAMS:
        kwargs["db_path"] = str(db_file)
    elif "database" in SQLITE_CTOR_PARAMS:
        kwargs["database"] = str(db_file)
    elif "path" in SQLITE_CTOR_PARAMS:
        kwargs["path"] = str(db_file)

    return VectorSearchIndex(**kwargs)

if SQLITESEARCH_AVAILABLE:
    try:
        t0 = time.perf_counter()

        for artifact_type in retrieval_types:
            cached = embedding_cache.get(artifact_type)
            if cached is None:
                continue

            type_df = artifacts_df[artifacts_df["Type"] == artifact_type].copy().reset_index(drop=True)
            payload = type_df[document_columns].fillna("").to_dict(orient="records")
            vectors = np.asarray(cached["embeddings"], dtype=np.float32)

            idx = make_sqlite_index(artifact_type)

            if REBUILD_SQLITE_VECTOR_INDEX:
                if hasattr(idx, "clear"):
                    try:
                        idx.clear()
                    except Exception:
                        pass
                idx.fit(vectors, payload)

            sqlite_vector_indexes[artifact_type] = idx

        sqlite_vector_build_seconds = time.perf_counter() - t0

        smoke_type = next(iter(sqlite_vector_indexes))
        smoke_q = embedding_model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]

        search_kwargs = {}
        if "num_results" in SQLITE_SEARCH_PARAMS:
            search_kwargs["num_results"] = 1
        elif "limit" in SQLITE_SEARCH_PARAMS:
            search_kwargs["limit"] = 1
        else:
            raise TypeError("Installed VectorSearchIndex.search exposes neither num_results nor limit.")
        if "output_ids" in SQLITE_SEARCH_PARAMS:
            search_kwargs["output_ids"] = True

        smoke = sqlite_vector_indexes[smoke_type].search(
            np.asarray(smoke_q, dtype=np.float32), **search_kwargs
        )
        if len(smoke) < 1:
            raise RuntimeError("SQLite vector smoke test returned no result.")

        SQLITE_VECTOR_SMOKE_OK = True
        print("SQLite build/search smoke test: PASS")
        print(f"Build/open time: {sqlite_vector_build_seconds:.4f} seconds")

    except Exception as exc:
        SQLITE_VECTOR_ERROR = repr(exc)
        sqlite_vector_indexes = {}
        print("SQLite build/search smoke test: FAIL")
        print(SQLITE_VECTOR_ERROR)
        print("SQLite will be excluded from evaluation.")
else:
    print("sqlitesearch unavailable; experiment skipped.")


In [ ]:
def sqlite_vector_retrieve_by_type(query_text, artifact_type, limit, model=embedding_model):
    if not SQLITE_VECTOR_SMOKE_OK:
        return []

    idx = sqlite_vector_indexes.get(artifact_type)
    if idx is None:
        return []

    qv = model.encode([query_text], normalize_embeddings=True, show_progress_bar=False)[0]

    search_kwargs = {}
    if "num_results" in SQLITE_SEARCH_PARAMS:
        search_kwargs["num_results"] = min(limit, len(documents_by_type.get(artifact_type, [])))
    elif "limit" in SQLITE_SEARCH_PARAMS:
        search_kwargs["limit"] = min(limit, len(documents_by_type.get(artifact_type, [])))
    else:
        return []
    if "output_ids" in SQLITE_SEARCH_PARAMS:
        search_kwargs["output_ids"] = True

    results = idx.search(np.asarray(qv, dtype=np.float32), **search_kwargs)

    output = []
    for rank, item in enumerate(results, start=1):
        row = dict(item)
        raw_score = row.get("score", row.get("_score", np.nan))
        row["vector_similarity"] = float(raw_score) if pd.notna(raw_score) else np.nan
        row["vector_rank"] = rank
        row["retrieval_source"] = "SQLite vector"
        output.append(row)
    return output

sqlite_vector_results_by_type = {}
sqlite_query_seconds = np.nan

if SQLITE_VECTOR_SMOKE_OK:
    t0 = time.perf_counter()
    for artifact_type in retrieval_types:
        retain_k = PER_TYPE_CANDIDATES.get(artifact_type, DEFAULT_PER_TYPE_CANDIDATES)
        sqlite_vector_results_by_type[artifact_type] = sqlite_vector_retrieve_by_type(
            query, artifact_type, retain_k
        )
    sqlite_query_seconds = time.perf_counter() - t0
    print(f"Current-query SQLite retrieval: {sqlite_query_seconds:.4f} seconds")
else:
    print("SQLite retrieval skipped: smoke test did not pass.")


## 8C. Optional Embedding-Model Comparison

The original notebook deliberately kept `all-MiniLM-L6-v2` as the baseline until retrieval quality could be measured. This experiment makes that decision measurable.

The comparison is **disabled by default** because each additional model must be downloaded and the full artifact corpus must be embedded again. Set `RUN_MODEL_COMPARISON = True` when you want to run the experiment.

The default challenger is `BAAI/bge-small-en-v1.5`. The experiment does not assume that it is better; the winner is determined from TraceGuard's ground truth.


In [ ]:
RUN_MODEL_COMPARISON = False

EMBEDDING_MODELS_TO_COMPARE = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
]

model_embedding_caches = {
    EMBEDDING_MODEL_NAME: embedding_cache
}
model_objects = {
    EMBEDDING_MODEL_NAME: embedding_model
}
model_build_times = {
    EMBEDDING_MODEL_NAME: np.nan
}

if RUN_MODEL_COMPARISON:
    for model_name in EMBEDDING_MODELS_TO_COMPARE:
        if model_name in model_embedding_caches:
            continue

        print(f"Encoding corpus with {model_name} ...")
        t0 = time.perf_counter()
        model = SentenceTransformer(model_name)
        cache = {}

        for artifact_type in retrieval_types:
            type_df = artifacts_df[
                artifacts_df["Type"] == artifact_type
            ].copy()

            texts = type_df["Search_Text"].tolist()
            if not texts:
                continue

            vectors = model.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
            )

            cache[artifact_type] = {
                "ids": type_df["ID"].astype(str).tolist(),
                "embeddings": np.asarray(vectors, dtype=np.float32),
            }

        model_objects[model_name] = model
        model_embedding_caches[model_name] = cache
        model_build_times[model_name] = time.perf_counter() - t0

        print(
            f"{model_name}: corpus encoded in "
            f"{model_build_times[model_name]:.4f} seconds"
        )
else:
    print("Embedding-model comparison is disabled.")
    print("Set RUN_MODEL_COMPARISON = True and rerun this cell to compare models.")


## 8D. Runtime Retrieval Strategy

The experimental indexes above are evaluated separately. The downstream TraceGuard pipeline remains stable by default and continues to use the original **lexical + brute-force semantic** candidate generation.

After batch evaluation, you may change `RUNTIME_VECTOR_BACKEND` to adopt another validated vector backend without rewriting traceability or LLM logic.


In [ ]:
# Safe default: preserves the exact semantic behavior of 02-basic-rag.ipynb.
# Allowed values: "brute_force", "minsearch", "sqlite"
RUNTIME_VECTOR_BACKEND = "brute_force"

if RUNTIME_VECTOR_BACKEND == "minsearch":
    if not MINSEARCH_VECTOR_SMOKE_OK:
        raise RuntimeError(
            "RUNTIME_VECTOR_BACKEND='minsearch' requires minsearch.VectorSearch."
        )
    semantic_results_by_type = minsearch_vector_results_by_type

elif RUNTIME_VECTOR_BACKEND == "sqlite":
    if not SQLITE_VECTOR_SMOKE_OK:
        raise RuntimeError(
            "RUNTIME_VECTOR_BACKEND='sqlite' requires sqlitesearch."
        )
    semantic_results_by_type = sqlite_vector_results_by_type

elif RUNTIME_VECTOR_BACKEND != "brute_force":
    raise ValueError(
        "RUNTIME_VECTOR_BACKEND must be 'brute_force', 'minsearch', or 'sqlite'."
    )

print("Runtime vector backend:", RUNTIME_VECTOR_BACKEND)


## 9. Merge Lexical + Semantic Discovery Without Losing Evidence

An artifact can be found by several mechanisms. We aggregate evidence rather than using `.drop_duplicates(..., keep="first")` to overwrite it.


In [ ]:
def add_candidate_evidence(store, row, source, **evidence):
    artifact_id = str(row["ID"])

    if artifact_id not in store:
        base = {
            col: row.get(col, "")
            for col in document_columns
        }
        base.update({
            "retrieval_sources": set(),
            "traceability_paths": [],
            "traceability_relationships": set(),
            "semantic_similarity": np.nan,
            "lexical_score": np.nan,
            "semantic_rank": np.nan,
            "lexical_rank": np.nan,
            "traceability_distance": np.nan,
        })
        store[artifact_id] = base

    item = store[artifact_id]
    item["retrieval_sources"].add(source)

    for key, value in evidence.items():
        if key in {"semantic_similarity", "lexical_score"}:
            if value is not None and pd.notna(value):
                current = item.get(key, np.nan)
                item[key] = float(value) if pd.isna(current) else max(float(current), float(value))
        elif key in {"semantic_rank", "lexical_rank"}:
            if value is not None and pd.notna(value):
                current = item.get(key, np.nan)
                item[key] = int(value) if pd.isna(current) else min(int(current), int(value))
        else:
            item[key] = value


candidate_store = {}

for artifact_type, results in lexical_results_by_type.items():
    for row in results:
        add_candidate_evidence(
            candidate_store, row, "Lexical retrieval",
            lexical_score=row.get("lexical_score"),
            lexical_rank=row.get("lexical_rank"),
        )

for artifact_type, results in semantic_results_by_type.items():
    for row in results:
        add_candidate_evidence(
            candidate_store, row, "Semantic similarity",
            semantic_similarity=row.get("semantic_similarity"),
            semantic_rank=row.get("semantic_rank"),
        )

print("Unique independently discovered candidates:", len(candidate_store))


## 10. Build Traceability Graph from Spawns / Covers

Traceability is **supporting evidence**, not a relevance gate. Paths are stored so later explanations can be audited.

The parser below handles comma/semicolon-separated IDs and simple list-like strings. If your export uses a different relationship encoding, extend `parse_link_ids`.


In [ ]:
known_ids = set(artifacts_df["ID"].astype(str))
artifact_lookup = artifacts_df.set_index("ID", drop=False).to_dict(orient="index")

def parse_link_ids(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    text = str(value).strip()
    if not text:
        return []

    # Try Python-list-like values first.
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple, set)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    parts = re.split(r"[,;\n|]+", text)
    return [p.strip().strip("'\"") for p in parts if p.strip().strip("'\"")]


graph = defaultdict(list)
relationship_edges = set()

def add_edge(src, dst, relationship):
    if src in known_ids and dst in known_ids:
        graph[src].append((dst, relationship))
        graph[dst].append((src, f"reverse:{relationship}"))
        relationship_edges.add((src, dst, relationship))


for _, row in artifacts_df.iterrows():
    src = str(row["ID"])
    for dst in parse_link_ids(row.get("Spawns", "")):
        add_edge(src, dst, "Spawns")
    for dst in parse_link_ids(row.get("Covers", "")):
        add_edge(src, dst, "Covers")

print("Traceability edges:", len(relationship_edges))


## 11. Configurable Traceability Seeds + Controlled Multi-Hop Expansion

Traceability expansion no longer starts from every retrieved CR/PR. A separate `traceability_seed_candidates` set is selected using configurable rank and/or semantic-score criteria.

The defaults are **baseline settings only**. They must be calibrated from historical evaluation rather than treated as relevance truth. Traversal has a hop limit, path-level cycle prevention, and per-seed best-depth control. Edge direction and whether the relationship originated from `Spawns` or `Covers` are preserved.


In [ ]:
CHANGE_TYPES = {"Change Request", "Problem Report"}
MAX_TRACEABILITY_HOPS = 4

# Seed-selection policy: configurable and intended for evaluation/tuning.
# "rank", "similarity", or "rank_or_similarity".
TRACEABILITY_SEED_MODE = "rank"
TRACEABILITY_SEED_MAX_RANK = {
    "Change Request": 20,
    "Problem Report": 20,
}
TRACEABILITY_SEED_MIN_SIMILARITY = {
    "Change Request": None,
    "Problem Report": None,
}

def is_traceability_seed(row):
    artifact_type = row.get("Type")
    if artifact_type not in CHANGE_TYPES:
        return False

    rank_limit = TRACEABILITY_SEED_MAX_RANK.get(artifact_type)
    similarity_limit = TRACEABILITY_SEED_MIN_SIMILARITY.get(artifact_type)

    rank_ok = (
        rank_limit is not None
        and pd.notna(row.get("semantic_rank"))
        and int(row["semantic_rank"]) <= int(rank_limit)
    )
    similarity_ok = (
        similarity_limit is not None
        and pd.notna(row.get("semantic_similarity"))
        and float(row["semantic_similarity"]) >= float(similarity_limit)
    )

    if TRACEABILITY_SEED_MODE == "rank":
        return rank_ok
    if TRACEABILITY_SEED_MODE == "similarity":
        return similarity_ok
    if TRACEABILITY_SEED_MODE == "rank_or_similarity":
        return rank_ok or similarity_ok

    raise ValueError(f"Unknown TRACEABILITY_SEED_MODE: {TRACEABILITY_SEED_MODE}")


seed_rows = []
for artifact_id, item in candidate_store.items():
    if item.get("Type") in CHANGE_TYPES and is_traceability_seed(item):
        seed_rows.append({
            "ID": artifact_id,
            "Type": item.get("Type"),
            "semantic_similarity": item.get("semantic_similarity"),
            "semantic_rank": item.get("semantic_rank"),
            "lexical_rank": item.get("lexical_rank"),
        })

traceability_seed_candidates = pd.DataFrame(seed_rows)
seed_change_ids = set(
    traceability_seed_candidates["ID"].astype(str)
    if not traceability_seed_candidates.empty else []
)

def expand_traceability(seed_ids, max_hops=MAX_TRACEABILITY_HOPS):
    discoveries = defaultdict(list)

    for seed in seed_ids:
        # State includes current node + path. A node may be reached by different valid paths,
        # but paths cannot repeat a node, preventing reciprocal-edge cycles.
        queue = deque([(seed, [seed], [], 0)])
        best_distance = {seed: 0}

        while queue:
            current, path, edge_evidence, distance = queue.popleft()
            if distance >= max_hops:
                continue

            for neighbor, relationship in graph.get(current, []):
                if neighbor in path:
                    continue

                new_distance = distance + 1
                new_path = path + [neighbor]
                new_edges = edge_evidence + [{
                    "from": current,
                    "to": neighbor,
                    "relationship": relationship,
                }]

                discoveries[neighbor].append({
                    "seed_change_id": seed,
                    "distance": new_distance,
                    "path": new_path,
                    "edges": new_edges,
                })

                # Traverse a node again only if this seed found it at a shorter depth.
                if new_distance < best_distance.get(neighbor, math.inf):
                    best_distance[neighbor] = new_distance
                    queue.append((neighbor, new_path, new_edges, new_distance))

    return discoveries


traceability_discoveries = expand_traceability(seed_change_ids)

for artifact_id, paths in traceability_discoveries.items():
    if artifact_id not in artifact_lookup:
        continue

    row = artifact_lookup[artifact_id]
    min_distance = min(p["distance"] for p in paths)

    add_candidate_evidence(
        candidate_store,
        row,
        "Traceability expansion",
        traceability_distance=min_distance,
    )

    item = candidate_store[artifact_id]
    for p in paths:
        item["traceability_paths"].append(p)
        item["traceability_relationships"].update(
            edge["relationship"] for edge in p["edges"]
        )

print("Traceability seeds:", len(seed_change_ids))
print("Candidates after controlled traceability expansion:", len(candidate_store))
print("Unique artifacts discovered through traceability:", len(traceability_discoveries))


## 12. Candidate Evidence Table + Explicit Unlinked-Relevant Detection

`unlinked_relevant` means the artifact was discovered by semantic/lexical similarity but has no traceability discovery from the selected relevant CR/PR seeds. **No link found is not the same as not relevant.**


In [ ]:
candidate_rows = []

for artifact_id, item in candidate_store.items():
    paths = item.get("traceability_paths", [])
    sources = set(item.get("retrieval_sources", set()))

    discovered_by_similarity = bool(
        {"Semantic similarity", "Lexical retrieval"} & sources
    )
    discovered_by_traceability = "Traceability expansion" in sources

    row = dict(item)
    row["retrieval_sources"] = sorted(sources)
    row["traceability_relationships"] = sorted(item["traceability_relationships"])
    row["traceability_paths"] = paths
    row["traceability_hops"] = (
        min((p["distance"] for p in paths), default=np.nan)
    )
    row["traceability_status"] = (
        "Linked" if discovered_by_traceability else "No relevant link found"
    )
    row["discovered_by_similarity"] = discovered_by_similarity
    row["discovered_by_traceability"] = discovered_by_traceability
    row["unlinked_relevant"] = (
        discovered_by_similarity and not discovered_by_traceability
    )
    candidate_rows.append(row)

candidates_df = pd.DataFrame(candidate_rows)

candidates_df["has_semantic_evidence"] = candidates_df["semantic_similarity"].notna()
candidates_df["has_lexical_evidence"] = candidates_df["lexical_score"].notna()
candidates_df["has_traceability_evidence"] = candidates_df["discovered_by_traceability"]

print("Candidate count:", len(candidates_df))
print("Unlinked similarity candidates:", int(candidates_df["unlinked_relevant"].sum()))

display(
    candidates_df[
        [
            "ID", "Type", "Summary", "semantic_similarity", "semantic_rank",
            "lexical_score", "lexical_rank", "retrieval_sources",
            "traceability_status", "traceability_hops", "unlinked_relevant"
        ]
    ].head(30)
)


## 13. Candidate Ranking Layer — Ordering Only

`review_rank_score` is retained strictly as a **candidate ordering heuristic**. It is not impact probability, calibrated confidence, or an impact decision. Semantic, lexical, and traceability signals remain available independently.


In [ ]:
def reciprocal_rank(rank):
    return 0.0 if pd.isna(rank) else 1.0 / float(rank)

def review_rank(row):
    semantic_rr = reciprocal_rank(row.get("semantic_rank", np.nan))
    lexical_rr = reciprocal_rank(row.get("lexical_rank", np.nan))
    trace_signal = 1.0 if row.get("traceability_status") == "Linked" else 0.0

    # Ranking heuristic only. Keep raw signals separately for evaluation/LLM.
    return semantic_rr + lexical_rr + 0.10 * trace_signal

candidates_df["review_rank_score"] = candidates_df.apply(review_rank, axis=1)

# Category is evidence-oriented, not "impacted/not impacted".
def evidence_category(row):
    text_signal = row["has_semantic_evidence"] or row["has_lexical_evidence"]
    trace_signal = row["has_traceability_evidence"]

    if text_signal and trace_signal:
        return "Strong candidate — similarity + traceability"
    if text_signal and not trace_signal:
        return "Similar candidate — no traceability found"
    if trace_signal and not text_signal:
        return "Traceability candidate — weak textual similarity"
    return "Low-confidence candidate"

candidates_df["candidate_category"] = candidates_df.apply(evidence_category, axis=1)

ranked_candidates_df = candidates_df.sort_values(
    ["review_rank_score", "semantic_similarity"],
    ascending=[False, False],
    na_position="last"
).reset_index(drop=True)

display(
    ranked_candidates_df[
        [
            "ID", "Type", "candidate_category", "semantic_similarity",
            "lexical_score", "traceability_distance", "retrieval_sources",
            "review_rank_score"
        ]
    ].head(40)
)


## 14. LLM Context Selection with Lifecycle + Unlinked Candidate Protection

`LLM_CONTEXT_LIMIT` remains independent of retrieval size. A portion of the downstream context is explicitly reserved for strong similarity-discovered **unlinked** artifacts so large traceability clusters cannot crowd them out.


In [ ]:
# First reserve context capacity for high-ranked unlinked similarity candidates.
unlinked_budget = min(
    int(round(LLM_CONTEXT_LIMIT * UNLINKED_CONTEXT_FRACTION)),
    int(ranked_candidates_df["unlinked_relevant"].sum())
)

unlinked_selected = ranked_candidates_df[
    ranked_candidates_df["unlinked_relevant"]
].head(unlinked_budget).copy()

selected_ids = set(unlinked_selected["ID"].astype(str))
remaining_budget = LLM_CONTEXT_LIMIT - len(unlinked_selected)

# Lifecycle-balanced allocation over the remaining budget.
selected_parts = [unlinked_selected] if not unlinked_selected.empty else []

if retrieval_types and remaining_budget > 0:
    per_type_budget = max(1, remaining_budget // len(retrieval_types))

    for artifact_type in retrieval_types:
        part = ranked_candidates_df[
            (ranked_candidates_df["Type"] == artifact_type)
            & (~ranked_candidates_df["ID"].astype(str).isin(selected_ids))
        ].head(per_type_budget)

        selected_parts.append(part)
        selected_ids.update(part["ID"].astype(str))

selected_candidates = (
    pd.concat(selected_parts, ignore_index=True)
    if selected_parts else pd.DataFrame(columns=ranked_candidates_df.columns)
)

# Fill any remaining slots globally by review ordering.
remaining = LLM_CONTEXT_LIMIT - len(selected_candidates)
if remaining > 0:
    already = set(selected_candidates["ID"].astype(str))
    extra = ranked_candidates_df[
        ~ranked_candidates_df["ID"].astype(str).isin(already)
    ].head(remaining)
    selected_candidates = pd.concat(
        [selected_candidates, extra], ignore_index=True
    )

selected_candidates = (
    selected_candidates
    .drop_duplicates(subset=["ID"])
    .sort_values("review_rank_score", ascending=False)
    .head(LLM_CONTEXT_LIMIT)
    .reset_index(drop=True)
)

print("Artifacts selected for grounded review:", len(selected_candidates))
print("Unlinked candidates protected in context:",
      int(selected_candidates["unlinked_relevant"].sum()))
display(selected_candidates["Type"].value_counts().to_frame("selected"))


## 15. Build Structured Grounded LLM Context


In [ ]:
def format_trace_paths(paths, max_paths=10):
    formatted = []
    for p in paths[:max_paths]:
        formatted.append({
            "path": " -> ".join(p["path"]),
            "edges": p["edges"],
            "distance": p["distance"],
            "seed_change_id": p["seed_change_id"],
        })
    return formatted

context_records = []

for _, row in selected_candidates.iterrows():
    context_records.append({
        "artifact_id": str(row["ID"]),
        "artifact_type": str(row["Type"]),
        "state": str(row.get("State", "")),
        "project": str(row.get("Project", "")),
        "summary": str(row.get("Summary", "")),
        "text": str(row.get("Text", "")),
        "semantic_similarity": (
            None if pd.isna(row.get("semantic_similarity"))
            else round(float(row["semantic_similarity"]), 4)
        ),
        "semantic_rank": (
            None if pd.isna(row.get("semantic_rank"))
            else int(row["semantic_rank"])
        ),
        "lexical_score": (
            None if pd.isna(row.get("lexical_score"))
            else round(float(row["lexical_score"]), 4)
        ),
        "lexical_rank": (
            None if pd.isna(row.get("lexical_rank"))
            else int(row["lexical_rank"])
        ),
        "retrieval_sources": row["retrieval_sources"],
        "traceability_status": row["traceability_status"],
        "traceability_hops": (
            None if pd.isna(row.get("traceability_hops"))
            else int(row["traceability_hops"])
        ),
        "traceability_paths": format_trace_paths(row["traceability_paths"]),
        "unlinked_relevant": bool(row["unlinked_relevant"]),
        "candidate_category": row["candidate_category"],
        # Ordering signal only; not probability/confidence.
        "review_rank_score": round(float(row["review_rank_score"]), 6),
    })

impact_context_json = json.dumps(context_records, indent=2, ensure_ascii=False)

print("Context records:", len(context_records))
print(impact_context_json[:5000])


## 16. Structured JSON Prompt


In [ ]:
prompt = f"""
You are TraceGuard AI, an engineering change impact analysis reviewer.

PROPOSED CHANGE:
{query}

CANDIDATE ARTIFACTS:
{impact_context_json}

The retrieval pipeline has already discovered these candidates using independent
per-artifact lexical retrieval, semantic similarity, and/or traceability expansion.
Your job is to ASSESS the supplied candidates, not discover new artifacts.

CRITICAL FIRST STEP — INPUT RELEVANCE:

Before performing ANY impact assessment, classify the PROPOSED CHANGE as
either "Relevant" or "Irrelevant".

A proposed change is Relevant only if its actual subject and intent concern
the engineering/system/product domain represented by the candidate artifacts.

Do NOT classify an input as Relevant merely because retrieved candidates have
some semantic or lexical similarity. Retrieval always returns candidates and
those candidates may be false matches.

For example, questions about education, classes, travel, food, personal
activities, entertainment, or other unrelated topics are Irrelevant.

If the input is Irrelevant:
- set "input_relevance" to "Irrelevant"
- return "assessments": []
- do not assess any candidate artifact
- do not infer an engineering interpretation of the input
- set "overall_assessment" to:
  "The input is unrelated to the available engineering artifacts. No impact analysis was performed."

Only if input_relevance is "Relevant" should you continue with the impact
assessment rules below.

Rules:
1. Use only artifact IDs and evidence present in CANDIDATE ARTIFACTS.
2. Never invent IDs, relationships, paths, scores, releases, tasks, or engineering facts.
3. Similarity and traceability are separate evidence signals; neither alone proves impact.
4. A high-similarity artifact with "No relevant link found" must not be discarded solely
   because it is unlinked. Surface it for engineering review when appropriate.
5. Do not impose a hard cosine-similarity threshold.
6. Treat "No relevant link found" as absence of discovered traceability, NOT as evidence of irrelevance.
7. review_rank_score is ordering only; never interpret it as impact probability or confidence.
8. If you cite a traceability relationship/path, reproduce only a supplied path.
9. Confidence is your review confidence from the supplied evidence, not a calibrated probability.
10. Prefer recall: include plausible candidates, but clearly communicate uncertainty.
11. Return valid JSON only. No markdown fences and no prose outside the JSON.

Return this exact top-level structure:
{{
  "proposed_change": "...",
  "input_relevance": "Relevant | Irrelevant",
  "assessments": [
    {{
      "artifact_id": "...",
      "artifact_type": "...",
      "impact_level": "High | Medium | Low | Review",
      "candidate_category": "Strong candidate — similarity + traceability | Similar candidate — no traceability found | Traceability candidate — weak textual similarity | Low-confidence candidate",
      "reason": "...",
      "evidence": {{
        "semantic_similarity": null,
        "lexical_score": null,
        "retrieval_sources": [],
        "traceability_paths": []
      }},
      "traceability_status": "Linked | No relevant link found",
      "confidence": "High | Medium | Low"
    }}
  ],
  "overall_assessment": "..."
}}
""".strip()

print(prompt[:5000])


## 17. Call OpenAI and Parse JSON


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input=prompt
)

raw_output = response.output_text.strip()

try:
    impact_assessment = json.loads(raw_output)
except json.JSONDecodeError as exc:
    print(raw_output)
    raise ValueError("LLM did not return valid JSON.") from exc

impact_assessment


## 18. Validate IDs, Paths, and “Linked” Claims

Validation now checks:
- every LLM artifact ID exists in supplied context;
- every claimed traceability path was supplied;
- any artifact labeled `Linked` by the LLM is actually traceability-linked in source-derived candidate evidence;
- supplied path edges correspond to graph relationships built from source `Spawns` / `Covers` data.


In [ ]:
context_ids = {record["artifact_id"] for record in context_records}
context_by_id = {record["artifact_id"]: record for record in context_records}

allowed_paths_by_id = defaultdict(set)
allowed_path_edges_by_id = defaultdict(dict)

for record in context_records:
    for p in record["traceability_paths"]:
        allowed_paths_by_id[record["artifact_id"]].add(p["path"])
        allowed_path_edges_by_id[record["artifact_id"]][p["path"]] = p.get("edges", [])

assessment_ids = {
    str(item.get("artifact_id", ""))
    for item in impact_assessment.get("assessments", [])
}

unknown_ids = assessment_ids - context_ids
invalid_relationship_claims = []
invalid_linked_claims = []
invalid_source_edges = []

# Validate the source-derived paths themselves against the graph edge evidence.
for artifact_id, path_map in allowed_path_edges_by_id.items():
    for path_text, edges in path_map.items():
        for edge in edges:
            src = str(edge.get("from", ""))
            dst = str(edge.get("to", ""))
            relationship = str(edge.get("relationship", ""))

            graph_matches = {
                (n, rel) for n, rel in graph.get(src, [])
            }
            if (dst, relationship) not in graph_matches:
                invalid_source_edges.append({
                    "artifact_id": artifact_id,
                    "path": path_text,
                    "edge": edge,
                })

for item in impact_assessment.get("assessments", []):
    artifact_id = str(item.get("artifact_id", ""))
    evidence = item.get("evidence", {}) or {}
    claimed_paths = evidence.get("traceability_paths", []) or []

    for claimed in claimed_paths:
        if isinstance(claimed, dict):
            path_text = str(claimed.get("path", ""))
        else:
            path_text = str(claimed)

        if path_text and path_text not in allowed_paths_by_id.get(artifact_id, set()):
            invalid_relationship_claims.append({
                "artifact_id": artifact_id,
                "claimed_path": path_text,
            })

    llm_status = str(item.get("traceability_status", ""))
    source_status = context_by_id.get(artifact_id, {}).get("traceability_status")

    if llm_status == "Linked" and source_status != "Linked":
        invalid_linked_claims.append({
            "artifact_id": artifact_id,
            "llm_status": llm_status,
            "source_status": source_status,
        })

print("LLM artifact IDs:", len(assessment_ids))
print("Unsupported artifact IDs:", sorted(unknown_ids))
print("Invalid relationship/path claims:", invalid_relationship_claims)
print("Invalid LLM 'Linked' claims:", invalid_linked_claims)
print("Invalid source-derived path edges:", invalid_source_edges)

if (
    not unknown_ids
    and not invalid_relationship_claims
    and not invalid_linked_claims
    and not invalid_source_edges
):
    print("Grounding and relationship validation passed.")


## 19. Determine Affected Release-Specific Baselines

Baseline impact is derived **deterministically from source traceability**, not invented
by the LLM.

The rule is:

**assessed relevant CR/PR → Release that Covers it → release-specific baseline → assessed ALM artifacts present in that baseline**

An ALM artifact may exist in several historical baselines. That does **not** mean every
baseline is affected. A baseline is surfaced only when its Release has explicit
traceability to a relevant CR/PR and the assessed ALM artifact is a member of that
Release baseline.

If no relevant CR/PR can be mapped to a Release, TraceGuard reports that the affected
baseline cannot be determined.


In [ ]:
ALM_BASELINE_TYPES = {
    "ALM Input",
    "ALM Requirement",
    "ALM Specification",
    "ALM Test Suite",
    "ALM Test Case",
}
CHANGE_RECORD_TYPES = {"Change Request", "Problem Report"}

# High/Medium are used for deterministic baseline impact.
# "Review" remains visible to the engineer but does not by itself establish
# a release-specific baseline impact.
BASELINE_IMPACT_LEVELS = {"High", "Medium"}

assessment_by_id = {
    str(item.get("artifact_id", "")): item
    for item in impact_assessment.get("assessments", [])
}

assessed_impacted_alm_ids = {
    artifact_id
    for artifact_id, item in assessment_by_id.items()
    if str(item.get("artifact_type", "")) in ALM_BASELINE_TYPES
    and str(item.get("impact_level", "")) in BASELINE_IMPACT_LEVELS
}

assessed_relevant_change_ids = {
    artifact_id
    for artifact_id, item in assessment_by_id.items()
    if str(item.get("artifact_type", "")) in CHANGE_RECORD_TYPES
    and str(item.get("impact_level", "")) in BASELINE_IMPACT_LEVELS
}

# Release -> explicitly covered CR/PR IDs, using only source data.
release_rows = artifacts_df[artifacts_df["Type"] == "Release"].copy()
release_to_changes = {}

for _, release_row in release_rows.iterrows():
    release_id = str(release_row["ID"])
    release_to_changes[release_id] = set(parse_link_ids(release_row["Covers"]))

# Baseline membership lookup.
baseline_members = defaultdict(set)

for _, baseline_row in baselines_df.iterrows():
    release_id = str(baseline_row["Release_ID"])
    artifact_id = str(baseline_row["Artifact_ID"])
    baseline_members[release_id].add(artifact_id)

baseline_impact_rows = []

for release_id, covered_changes in release_to_changes.items():
    supporting_changes = sorted(
        assessed_relevant_change_ids.intersection(covered_changes)
    )

    if not supporting_changes:
        continue

    impacted_members = sorted(
        assessed_impacted_alm_ids.intersection(
            baseline_members.get(release_id, set())
        )
    )

    if not impacted_members:
        continue

    baseline_id = f"BL-{release_id}"

    baseline_impact_rows.append({
        "Release_ID": release_id,
        "Baseline_ID": baseline_id,
        "Supporting_CR_PR_IDs": ", ".join(supporting_changes),
        "Affected_ALM_Artifact_IDs": ", ".join(impacted_members),
        "Supporting_CR_PR_Count": len(supporting_changes),
        "Affected_ALM_Count": len(impacted_members),
        "Determination": "Traceability-supported baseline impact",
    })

affected_baselines_df = pd.DataFrame(baseline_impact_rows)

if affected_baselines_df.empty:
    baseline_determination = {
        "status": "Undetermined",
        "reason": (
            "No assessed High/Medium CR or PR could be connected through an "
            "explicit Release Covers relationship to a Release baseline that "
            "also contains assessed High/Medium ALM artifacts."
        ),
        "affected_release_ids": [],
        "affected_baseline_ids": [],
    }

    print("Affected baseline: UNDETERMINED")
    print(baseline_determination["reason"])
else:
    affected_baselines_df = affected_baselines_df.sort_values(
        ["Supporting_CR_PR_Count", "Affected_ALM_Count"],
        ascending=False,
    ).reset_index(drop=True)

    baseline_determination = {
        "status": "Determined",
        "reason": "Explicit CR/PR -> Release traceability plus ALM baseline membership.",
        "affected_release_ids": affected_baselines_df["Release_ID"].tolist(),
        "affected_baseline_ids": affected_baselines_df["Baseline_ID"].tolist(),
    }

    print("Traceability-supported affected Release/Baseline candidates:")
    display(affected_baselines_df)

baseline_determination


## 20. Human-Readable Impact Review

The JSON remains the system-of-record output. A readable table can be generated afterward without losing structured evidence.


In [ ]:
assessment_rows = []

for item in impact_assessment.get("assessments", []):
    assessment_rows.append({
        "artifact_id": item.get("artifact_id"),
        "artifact_type": item.get("artifact_type"),
        "impact_level": item.get("impact_level"),
        "candidate_category": item.get("candidate_category"),
        "traceability_status": item.get("traceability_status"),
        "confidence": item.get("confidence"),
        "reason": item.get("reason"),
    })

impact_report_df = pd.DataFrame(assessment_rows)

if not impact_report_df.empty:
    display(
        impact_report_df.sort_values(
            ["impact_level", "confidence"],
            ascending=[True, True]
        )
    )

print("\nOverall assessment:")
print(impact_assessment.get("overall_assessment", ""))


## 21. Evaluation Ground Truth + Retrieval Calibration

The final synthetic data model keeps the unseen CR text and its answer key in
**separate files**:

- `evaluation_new_crs.csv` → unseen incoming CRs
- `evaluation_ground_truth.csv` → affected ALM artifacts, related CR/PR records,
  affected Release, and affected Baseline

Retrieval evaluation therefore reads the dedicated ground-truth file directly.
Semantic threshold sweeps continue to use the **complete scored artifact population**,
not only retained Top-K candidates.


In [ ]:
RECALL_K_VALUES = (10, 25, 50, 75, 100)
SIMILARITY_THRESHOLDS = (0.40, 0.50, 0.60, 0.70, 0.80)

def split_ids(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    return [x.strip() for x in str(value).split(",") if x.strip()]

ground_truth_rows = []

for _, row in evaluation_ground_truth_df.iterrows():
    eval_id = str(row["Evaluation_CR_ID"]).strip()

    for affected_id in split_ids(row["Affected_Artifact_IDs"]):
        ground_truth_rows.append({
            "evaluation_cr_id": eval_id,
            "affected_artifact_id": affected_id,
            "affected_artifact_type": str(
                artifact_lookup.get(affected_id, {}).get("Type", "")
            ),
            "affected_release_id": (
                "" if pd.isna(row.get("Affected_Release_ID"))
                else str(row.get("Affected_Release_ID", "")).strip()
            ),
            "affected_baseline_id": (
                "" if pd.isna(row.get("Affected_Baseline_ID"))
                else str(row.get("Affected_Baseline_ID", "")).strip()
            ),
            "baseline_determinable": bool(row.get("Baseline_Determinable", False)),
        })

ground_truth_df = pd.DataFrame(ground_truth_rows).drop_duplicates()

print("Ground-truth artifact mappings:", len(ground_truth_df))
display(ground_truth_df.head(20))


def recall_at_k(ranked_ids, relevant_ids, k):
    relevant_ids = set(map(str, relevant_ids))
    if not relevant_ids:
        return np.nan
    retrieved = set(map(str, ranked_ids[:k]))
    return len(retrieved & relevant_ids) / len(relevant_ids)


def precision_at_k(ranked_ids, relevant_ids, k):
    relevant_ids = set(map(str, relevant_ids))
    retrieved = list(map(str, ranked_ids[:k]))
    if not retrieved:
        return np.nan
    return len(set(retrieved) & relevant_ids) / len(set(retrieved))


def evaluate_current_query_by_type(
    all_scored_by_type,
    relevant_ids,
    ks=RECALL_K_VALUES,
    thresholds=SIMILARITY_THRESHOLDS,
):
    relevant_ids = set(map(str, relevant_ids))
    rows = []

    for artifact_type, scored_df in all_scored_by_type.items():
        if scored_df.empty:
            continue

        type_relevant = {
            rid for rid in relevant_ids
            if str(artifact_lookup.get(rid, {}).get("Type", "")) == artifact_type
        }
        ranked_ids = scored_df["ID"].astype(str).tolist()

        record = {
            "artifact_type": artifact_type,
            "ground_truth_count": len(type_relevant),
        }

        for k in ks:
            record[f"Recall@{k}"] = recall_at_k(ranked_ids, type_relevant, k)
            record[f"Precision@{k}"] = precision_at_k(ranked_ids, type_relevant, k)

        for threshold in thresholds:
            retrieved = set(
                scored_df.loc[
                    scored_df["semantic_similarity"] >= threshold, "ID"
                ].astype(str)
            )
            record[f"Recall@sim>={threshold:.2f}"] = (
                len(retrieved & type_relevant) / len(type_relevant)
                if type_relevant else np.nan
            )
            record[f"Precision@sim>={threshold:.2f}"] = (
                len(retrieved & type_relevant) / len(retrieved)
                if retrieved else np.nan
            )

        rows.append(record)

    return pd.DataFrame(rows)


def evaluate_current_query_overall(
    all_scored_by_type,
    relevant_ids,
    ks=RECALL_K_VALUES,
):
    relevant_ids = set(map(str, relevant_ids))

    combined = pd.concat(
        [df for df in all_scored_by_type.values() if not df.empty],
        ignore_index=True
    ).sort_values("semantic_similarity", ascending=False)

    ranked_ids = combined["ID"].astype(str).tolist()

    return pd.DataFrame([{
        "ground_truth_count": len(relevant_ids),
        **{f"Recall@{k}": recall_at_k(ranked_ids, relevant_ids, k) for k in ks},
        **{f"Precision@{k}": precision_at_k(ranked_ids, relevant_ids, k) for k in ks},
    }])


if EVALUATION_CR_ID:
    current_truth = ground_truth_df.loc[
        ground_truth_df["evaluation_cr_id"].astype(str) == str(EVALUATION_CR_ID),
        "affected_artifact_id"
    ].astype(str).tolist()

    overall_metrics_df = evaluate_current_query_overall(
        all_semantic_scores_by_type,
        current_truth,
    )
    by_type_metrics_df = evaluate_current_query_by_type(
        all_semantic_scores_by_type,
        current_truth,
    )

    print("Overall retrieval evaluation:")
    display(overall_metrics_df)

    print("Retrieval evaluation by artifact type:")
    display(by_type_metrics_df)

    gt_row = evaluation_ground_truth_df[
        evaluation_ground_truth_df["Evaluation_CR_ID"].astype(str)
        == str(EVALUATION_CR_ID)
    ].iloc[0]

    print("Ground-truth Release:", gt_row.get("Affected_Release_ID", ""))
    print("Ground-truth Baseline:", gt_row.get("Affected_Baseline_ID", ""))
    print("Baseline determinable:", gt_row.get("Baseline_Determinable", False))
else:
    print(
        "Evaluation framework is ready. Set EVALUATION_CR_ID near the incoming "
        "change cell (for example EVAL-CR-001) and run the notebook top-to-bottom."
    )


## 21A. Module 2 Batch Retrieval Evaluation — Corrected Per-Type Protocol

TraceGuard deliberately fits separate lexical indexes per lifecycle artifact type. Raw TF-IDF scores from those independently fitted spaces are **not globally calibrated**, so this notebook does not globally sort those scores.

Evaluation now uses:

1. **Per-artifact-type Recall@K and Precision@K**.
2. **Runtime-like candidate-pool recall and precision**: Top-K is retained independently per type and then unioned.
3. **Hybrid RRF within each artifact type**: lexical and semantic ranks are fused without mixing raw TF-IDF and cosine scores.
4. **Query latency**, reported separately.

This directly tests the calibration caveat instead of merely documenting it.


In [ ]:
BATCH_K_VALUES = (5, 10, 20, 50)
RRF_K = 60

def evaluation_query_text(eval_id):
    match = evaluation_df[evaluation_df["Evaluation_CR_ID"].astype(str) == str(eval_id)]
    if match.empty:
        raise ValueError(f"No evaluation query found for {eval_id}")
    return str(match.iloc[0]["Summary"]).strip()

def truth_ids_for_eval(eval_id):
    return ground_truth_df.loc[
        ground_truth_df["evaluation_cr_id"].astype(str) == str(eval_id),
        "affected_artifact_id"
    ].astype(str).tolist()

artifact_type_lookup = dict(zip(artifacts_df["ID"].astype(str), artifacts_df["Type"].astype(str)))

def truth_ids_by_type(eval_id):
    grouped = defaultdict(set)
    for artifact_id in truth_ids_for_eval(eval_id):
        artifact_type = artifact_type_lookup.get(str(artifact_id))
        if artifact_type is not None:
            grouped[artifact_type].add(str(artifact_id))
    return grouped

def lexical_rankings_by_type(query_text):
    rankings = {}
    for artifact_type in retrieval_types:
        n = len(documents_by_type.get(artifact_type, []))
        results = lexical_retrieve_by_type(query_text, artifact_type, n)
        rankings[artifact_type] = [str(x["ID"]) for x in results]
    return rankings

def brute_force_rankings_by_type(query_text, model=embedding_model, cache=embedding_cache):
    qv = model.encode([query_text], normalize_embeddings=True, show_progress_bar=False)[0]
    rankings = {}
    for artifact_type, cached in cache.items():
        sims = np.asarray(cached["embeddings"]) @ qv
        order = np.argsort(sims)[::-1]
        rankings[artifact_type] = [str(cached["ids"][j]) for j in order]
    return rankings

def minsearch_rankings_by_type(query_text):
    if not MINSEARCH_VECTOR_SMOKE_OK:
        return {}
    return {
        t: [str(x["ID"]) for x in minsearch_vector_retrieve_by_type(query_text, t, len(documents_by_type.get(t, [])))]
        for t in retrieval_types
    }

def sqlite_rankings_by_type(query_text):
    if not SQLITE_VECTOR_SMOKE_OK:
        return {}
    rankings = {}
    for t in retrieval_types:
        rows = sqlite_vector_retrieve_by_type(query_text, t, len(documents_by_type.get(t, [])))
        rankings[t] = [str(x.get("ID", x.get("id"))) for x in rows if x.get("ID", x.get("id")) is not None]
    return rankings

def reciprocal_rank_fusion(*ranked_lists, rrf_k=RRF_K):
    scores = defaultdict(float)
    for ranked in ranked_lists:
        seen = set()
        for rank, artifact_id in enumerate(ranked, start=1):
            artifact_id = str(artifact_id)
            if artifact_id in seen:
                continue
            seen.add(artifact_id)
            scores[artifact_id] += 1.0 / (rrf_k + rank)
    return [artifact_id for artifact_id, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

def hybrid_rrf_rankings_by_type(lexical_by_type, vector_by_type):
    return {
        t: reciprocal_rank_fusion(lexical_by_type.get(t, []), vector_by_type.get(t, []))
        for t in set(lexical_by_type) | set(vector_by_type)
    }

def candidate_pool(rankings_by_type, k_spec):
    pool = set()
    for t, ranked in rankings_by_type.items():
        k = k_spec.get(t, DEFAULT_PER_TYPE_CANDIDATES) if isinstance(k_spec, dict) else int(k_spec)
        pool.update(map(str, ranked[:k]))
    return pool

def pool_recall(pool, truth):
    truth = set(map(str, truth))
    return len(set(pool) & truth) / len(truth) if truth else np.nan

def pool_precision(pool, truth):
    pool, truth = set(map(str, pool)), set(map(str, truth))
    return len(pool & truth) / len(pool) if pool else np.nan

def per_type_rows(eval_id, strategy, rankings):
    truth_by_type = truth_ids_by_type(eval_id)
    rows = []
    for t in retrieval_types:
        relevant = truth_by_type.get(t, set())
        if not relevant:
            continue
        row = {"evaluation_cr_id":str(eval_id), "strategy":strategy, "artifact_type":t, "ground_truth_count":len(relevant)}
        for k in BATCH_K_VALUES:
            row[f"Recall@{k}"] = recall_at_k(rankings.get(t, []), relevant, k)
            row[f"Precision@{k}"] = precision_at_k(rankings.get(t, []), relevant, k)
        rows.append(row)
    return rows

def pool_row(eval_id, strategy, rankings, truth, elapsed):
    runtime_pool = candidate_pool(rankings, PER_TYPE_CANDIDATES)
    row = {
        "evaluation_cr_id":str(eval_id),
        "strategy":strategy,
        "candidate_pool_size":len(runtime_pool),
        "candidate_pool_recall":pool_recall(runtime_pool, truth),
        "candidate_pool_precision":pool_precision(runtime_pool, truth),
        "query_seconds":elapsed,
    }
    for k in BATCH_K_VALUES:
        p = candidate_pool(rankings, k)
        row[f"PoolRecall@{k}_per_type"] = pool_recall(p, truth)
        row[f"PoolPrecision@{k}_per_type"] = pool_precision(p, truth)
    return row


In [ ]:
pool_metric_rows = []
per_type_metric_rows = []
batch_rankings = {}

evaluation_ids = evaluation_df["Evaluation_CR_ID"].dropna().astype(str).drop_duplicates().tolist()

for eval_id in evaluation_ids:
    q = evaluation_query_text(eval_id)
    truth = truth_ids_for_eval(eval_id)
    batch_rankings[eval_id] = {}

    strategies = []

    t0 = time.perf_counter()
    lex = lexical_rankings_by_type(q)
    strategies.append(("Lexical", lex, time.perf_counter()-t0))

    t0 = time.perf_counter()
    brute = brute_force_rankings_by_type(q)
    strategies.append(("Brute-force vector", brute, time.perf_counter()-t0))

    if MINSEARCH_VECTOR_SMOKE_OK:
        t0 = time.perf_counter()
        mini = minsearch_rankings_by_type(q)
        strategies.append(("Minsearch vector", mini, time.perf_counter()-t0))

    if SQLITE_VECTOR_SMOKE_OK:
        t0 = time.perf_counter()
        sq = sqlite_rankings_by_type(q)
        strategies.append(("SQLite vector", sq, time.perf_counter()-t0))

    t0 = time.perf_counter()
    hybrid = hybrid_rrf_rankings_by_type(lex, brute)
    strategies.append(("Hybrid lexical + vector RRF", hybrid, time.perf_counter()-t0))

    for name, rankings, elapsed in strategies:
        batch_rankings[eval_id][name] = rankings
        pool_metric_rows.append(pool_row(eval_id, name, rankings, truth, elapsed))
        per_type_metric_rows.extend(per_type_rows(eval_id, name, rankings))

pool_metrics_df = pd.DataFrame(pool_metric_rows)
per_type_metrics_df = pd.DataFrame(per_type_metric_rows)

print("Runtime-like candidate-pool metrics")
display(pool_metrics_df)
print("\nPer-artifact-type metrics")
display(per_type_metrics_df)


In [ ]:
pool_cols = [c for c in pool_metrics_df.columns if c.startswith("PoolRecall@") or c.startswith("PoolPrecision@")]

pool_summary_df = (
    pool_metrics_df.groupby("strategy", as_index=False)
    .agg(
        evaluation_queries=("evaluation_cr_id","nunique"),
        mean_query_seconds=("query_seconds","mean"),
        mean_candidate_pool_size=("candidate_pool_size","mean"),
        mean_candidate_pool_recall=("candidate_pool_recall","mean"),
        mean_candidate_pool_precision=("candidate_pool_precision","mean"),
        **{f"mean_{c}":(c,"mean") for c in pool_cols}
    )
    .sort_values(["mean_candidate_pool_recall","mean_candidate_pool_precision"], ascending=[False,False])
    .reset_index(drop=True)
)
display(pool_summary_df)

if not per_type_metrics_df.empty:
    metric_cols = [c for c in per_type_metrics_df.columns if c.startswith("Recall@") or c.startswith("Precision@")]
    per_type_summary_df = (
        per_type_metrics_df.groupby(["strategy","artifact_type"], as_index=False)
        .agg(evaluation_queries=("evaluation_cr_id","nunique"), **{f"mean_{c}":(c,"mean") for c in metric_cols})
    )
    display(per_type_summary_df)
else:
    per_type_summary_df = pd.DataFrame()


## 21B. Embedding-Model Evaluation

When `RUN_MODEL_COMPARISON = True`, each embedding model is evaluated with the same brute-force cosine retrieval logic. This isolates **embedding quality** from **index implementation**.

That distinction is important:

- Changing Minsearch vs SQLite changes the retrieval/index architecture.
- Changing MiniLM vs BGE changes the vector representation.
- Hybrid fusion changes how lexical and semantic evidence are combined.

Only one variable should be credited for an observed improvement.


In [ ]:
embedding_model_metrics_df = pd.DataFrame()
embedding_model_summary_df = pd.DataFrame()

if RUN_MODEL_COMPARISON:
    rows = []
    for model_name, cache in model_embedding_caches.items():
        model = model_objects[model_name]
        for eval_id in evaluation_ids:
            q = evaluation_query_text(eval_id)
            truth = truth_ids_for_eval(eval_id)
            t0 = time.perf_counter()
            rankings = brute_force_rankings_by_type(q, model=model, cache=cache)
            rows.append(pool_row(eval_id, model_name, rankings, truth, time.perf_counter()-t0))

    embedding_model_metrics_df = pd.DataFrame(rows)
    embedding_model_summary_df = (
        embedding_model_metrics_df.groupby("strategy", as_index=False)
        .agg(
            evaluation_queries=("evaluation_cr_id","nunique"),
            mean_query_seconds=("query_seconds","mean"),
            mean_candidate_pool_size=("candidate_pool_size","mean"),
            mean_candidate_pool_recall=("candidate_pool_recall","mean"),
            mean_candidate_pool_precision=("candidate_pool_precision","mean"),
        )
        .sort_values(["mean_candidate_pool_recall","mean_candidate_pool_precision"], ascending=[False,False])
    )
    display(embedding_model_summary_df)
else:
    print("Embedding-model evaluation skipped. Set RUN_MODEL_COMPARISON = True to run it.")


## 21C. Performance Summary

Accuracy and runtime are reported separately. A persistent index can be architecturally preferable even when its Recall@K is unchanged, because the corpus does not need to be re-embedded and rebuilt on every application start.


In [ ]:
performance_df = pd.DataFrame([
    {"component":"Minsearch vector","smoke_test_passed":MINSEARCH_VECTOR_SMOKE_OK,
     "build_or_open_seconds":minsearch_vector_build_seconds,"current_query_seconds":minsearch_query_seconds,
     "persistent":False,"error":MINSEARCH_VECTOR_ERROR},
    {"component":"SQLite vector","smoke_test_passed":SQLITE_VECTOR_SMOKE_OK,
     "build_or_open_seconds":sqlite_vector_build_seconds,"current_query_seconds":sqlite_query_seconds,
     "persistent":True,"error":SQLITE_VECTOR_ERROR},
])
display(performance_df)


## 21D. Architecture Decision Helper

The notebook does **not** automatically replace TraceGuard's runtime retriever. It recommends a retrieval strategy from the measured batch results, while the explicit `RUNTIME_VECTOR_BACKEND` switch remains under developer control.

Interpret the results as follows:

- Higher Recall@K means fewer known affected artifacts are missed.
- Higher Precision@K means a larger fraction of the retrieved Top-K is actually affected.
- Similar vector metrics across brute-force and Minsearch indicate that the abstraction changed, not the embedding quality.
- Similar vector metrics with better persistence/startup behavior support SQLite as an architectural improvement.
- Better BGE metrics support changing the embedding representation.
- Better hybrid metrics support retaining lexical evidence alongside semantic retrieval.


In [ ]:
if not pool_summary_df.empty:
    best = pool_summary_df.iloc[0]
    architecture_decision = {
        "selection_basis":"highest mean runtime-like candidate-pool recall; precision secondary",
        "recommended_retrieval_strategy":best["strategy"],
        "mean_candidate_pool_recall":float(best["mean_candidate_pool_recall"]),
        "mean_candidate_pool_precision":float(best["mean_candidate_pool_precision"]),
        "minsearch_smoke_test_passed":MINSEARCH_VECTOR_SMOKE_OK,
        "sqlite_smoke_test_passed":SQLITE_VECTOR_SMOKE_OK,
        "runtime_vector_backend_currently_used":RUNTIME_VECTOR_BACKEND,
    }
    print(json.dumps(architecture_decision, indent=2))
else:
    architecture_decision = {}


## 22. Retrieval Diagnostics

Every run reports retained semantic/lexical counts, merged independent candidates, traceability seeds/additions, unlinked candidates, total pool size, and LLM context size.


In [ ]:
diagnostic_rows = []

for artifact_type in retrieval_types:
    semantic_ids = {
        str(x["ID"]) for x in semantic_results_by_type.get(artifact_type, [])
    }
    lexical_ids = {
        str(x["ID"]) for x in lexical_results_by_type.get(artifact_type, [])
    }
    merged_ids = semantic_ids | lexical_ids

    diagnostic_rows.append({
        "artifact_type": artifact_type,
        "all_semantic_scored": len(all_semantic_scores_by_type.get(artifact_type, [])),
        "semantic_retained": len(semantic_ids),
        "lexical_retained": len(lexical_ids),
        "merged_independent": len(merged_ids),
    })

retrieval_diagnostics_df = pd.DataFrame(diagnostic_rows)

display(retrieval_diagnostics_df)

trace_added_ids = {
    artifact_id
    for artifact_id, item in candidate_store.items()
    if "Traceability expansion" in item.get("retrieval_sources", set())
}

print(f"Traceability seeds: {len(seed_change_ids)}")
print(f"Added/discovered through traceability: {len(trace_added_ids)}")
print(f"Unlinked similarity candidates: {int(candidates_df['unlinked_relevant'].sum())}")
print(f"Final candidate pool: {len(candidates_df)}")
print(f"Sent to LLM: {len(selected_candidates)}")
print(f"LLM context limit: {LLM_CONTEXT_LIMIT}")


# Module 2 — Findings and TraceGuard Runtime Architecture

The Module 2 experiments are deliberately separated from the production decision. Use `batch_summary_df`, `performance_df`, and—when enabled—`embedding_model_summary_df` as the evidence for deciding what should move into `traceguard.py`.

```text
                         Incoming Change Request
                                  |
                  +---------------+---------------+
                  |                               |
                  v                               v
           Lexical Retrieval              Vector Retrieval
           (per artifact type)       (validated backend/model)
                  |                               |
                  +---------------+---------------+
                                  |
                                  v
                    Hybrid / Retrieval Evidence
                                  |
                                  v
                    Select CR/PR Traceability Seeds
                                  |
                                  v
                   Controlled Traceability Expansion
                         (Spawns / Covers)
                                  |
                                  v
                    Candidate Evidence + Ranking
                                  |
                                  v
                   Lifecycle-Balanced LLM Context
                                  |
                                  v
                         LLM Relevance Check
                                  |
                        +---------+---------+
                        |                   |
                        v                   v
                   Irrelevant            Relevant
                        |                   |
                        v                   v
                No Impact Analysis     Grounded LLM
                   Performed            Assessment
                                            |
                                            v
                                  Validate IDs and Paths
                                            |
                                            v
                              Relevant CR/PR → Release
                                       → BL-Release
                                            |
                                            v
                              Intersect Assessed ALM
                               with Baseline Members
                                            |
                                  +---------+---------+
                                  |                   |
                                  v                   v
                         Baseline Determined    No Release Mapping
                         from Traceability       → Undetermined
```

## What to infer from this notebook

1. **Retrieval quality:** compare Recall@K and Precision@K across strategies.
2. **Index architecture:** compare build/open/query timings and persistence.
3. **Embedding quality:** if the optional model experiment is enabled, compare models using the same retrieval algorithm.
4. **Hybrid value:** compare lexical + vector RRF against either retriever alone.
5. **Downstream safety:** retrieval remains candidate discovery; explicit traceability and grounded validation still determine engineering evidence.
6. **Production decision:** adopt only the configuration supported by the evaluation results.

### Runtime vs Evaluation

The **Relevant / Irrelevant** classification remains part of the runtime pipeline. If an incoming query is unrelated to the available engineering artifacts, TraceGuard returns it as `Irrelevant` and does not perform engineering impact assessment.

Ground-truth evaluation, embedding comparison, vector-backend comparison, and performance diagnostics are development/calibration activities. They should not be required for every production change-analysis request.


## Module 2 Trust Check

This final table makes the two corrections explicit: optional package APIs must pass a real smoke test, and lexical evaluation must avoid global sorting of separately calibrated TF-IDF spaces.


In [ ]:
trust_check = pd.DataFrame([
    {"check":"Minsearch installed API + real build/search smoke test",
     "passed":bool(MINSEARCH_VECTOR_SMOKE_OK),
     "details":"included in evaluation" if MINSEARCH_VECTOR_SMOKE_OK else (MINSEARCH_VECTOR_ERROR or "unavailable")},
    {"check":"SQLite installed API + real build/search smoke test",
     "passed":bool(SQLITE_VECTOR_SMOKE_OK),
     "details":"included in evaluation" if SQLITE_VECTOR_SMOKE_OK else (SQLITE_VECTOR_ERROR or "unavailable")},
    {"check":"Lexical cross-type calibration issue removed from architecture decision",
     "passed":True,
     "details":"per-type rankings + runtime-like candidate-pool metrics"},
    {"check":"Hybrid raw-score mixing avoided",
     "passed":True,
     "details":"RRF is performed within artifact type"},
])
display(trust_check)


## Ready-to-Run Checklist

Before running top-to-bottom:

- Keep `artifacts.csv`, `baselines.csv`, `evaluation_new_crs.csv`, and `evaluation_ground_truth.csv` in `../data`.
- Install/upgrade `minsearch`, `sqlitesearch`, and `sentence-transformers`.
- Keep `RUNTIME_VECTOR_BACKEND = "brute_force"` for the first validation run.
- Keep `RUN_MODEL_COMPARISON = False` for the first run; enable it afterward if you want MiniLM vs BGE results.
- Set `REBUILD_SQLITE_VECTOR_INDEX = True` after artifact or embedding changes; after a successful persistent build, set it to `False` to test reopen/query behavior.
- Set `EVALUATION_CR_ID` only when you want detailed inspection of one evaluation CR. Batch evaluation runs all evaluation CRs automatically.
- Configure your existing OpenAI client/environment exactly as in the Basic RAG notebook before executing the LLM cell.

The final architecture decision should be based on the generated evaluation tables rather than on a predetermined expectation that vector search must outperform the existing hybrid baseline.
